In [1]:
import pandas as pd
import numpy as np

# =====================================================
# Input file
# =====================================================
infile = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUR_ClinicalData_20260427.NO.DATES.bloodID.withBCRv2.xlsx"

df = pd.read_excel(infile)

# =====================================================
# Columns to use
# =====================================================
id_col = "Patient_BCR_Barcode"

cols_to_check = [
    "Histologic.type",
    "ER_Status",
    "PR_Status",
    "HER2_Status"
]

# Keep only required columns
df_sub = df[[id_col] + cols_to_check].copy()

# Clean empty strings / spaces
for col in [id_col] + cols_to_check:
    df_sub[col] = (
        df_sub[col]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA})
    )

# Remove rows without Patient_BCR_Barcode
df_sub = df_sub.dropna(subset=[id_col])

# =====================================================
# Collapse repeated rows by Patient_BCR_Barcode
# Rule:
# - if same non-missing value across rows: keep value
# - if different non-missing values: Conflict
# - if all missing: Missing
# =====================================================
def collapse_patient_values(x):
    vals = sorted(x.dropna().unique())

    if len(vals) == 0:
        return "Missing"
    elif len(vals) == 1:
        return vals[0]
    else:
        return "Conflict"

collapsed = (
    df_sub
    .groupby(id_col, as_index=False)
    .agg({col: collapse_patient_values for col in cols_to_check})
)

# =====================================================
# Standardize receptor values for molecular subtype
# =====================================================
def normalize_receptor_status(x):
    if pd.isna(x) or x == "Missing":
        return "Missing"
    if x == "Conflict":
        return "Conflict"

    x_clean = str(x).strip().lower()

    if x_clean in ["positive", "pos", "+", "1", "yes"]:
        return "Positive"
    elif x_clean in ["negative", "neg", "-", "0", "no"]:
        return "Negative"
    else:
        return "Other"

for col in ["ER_Status", "PR_Status", "HER2_Status"]:
    collapsed[col + "_clean"] = collapsed[col].apply(normalize_receptor_status)

# =====================================================
# Derive Molecular_subtype
# HR+ = ER+ or PR+
# HR- = ER- and PR-
# HER2+ = HER2 positive
# HER2- = HER2 negative
# =====================================================
def derive_molecular_subtype(row):
    er = row["ER_Status_clean"]
    pr = row["PR_Status_clean"]
    her2 = row["HER2_Status_clean"]

    # Any conflict in receptor status
    if "Conflict" in [er, pr, her2]:
        return "Conflict"

    # Missing/Other values
    if er in ["Missing", "Other"] or pr in ["Missing", "Other"] or her2 in ["Missing", "Other"]:
        return "Unknown"

    # HR status
    if er == "Positive" or pr == "Positive":
        hr = "HR+"
    elif er == "Negative" and pr == "Negative":
        hr = "HR-"
    else:
        return "Unknown"

    # HER2 status
    if her2 == "Positive":
        her2_status = "HER2+"
    elif her2 == "Negative":
        her2_status = "HER2-"
    else:
        return "Unknown"

    return f"{hr}/{her2_status}"

collapsed["Molecular_subtype"] = collapsed.apply(derive_molecular_subtype, axis=1)

# =====================================================
# Print summaries
# =====================================================
print("\n================ UNIQUE CASE SUMMARY ================")
print(f"Number of unique Patient_BCR_Barcode: {collapsed[id_col].nunique()}")

print("\n================ HISTOLOGICAL SUBTYPE DISTRIBUTION ================")
print(collapsed["Histologic.type"].value_counts(dropna=False))

print("\n================ MOLECULAR SUBTYPE DISTRIBUTION ================")
molecular_order = [
    "HR+/HER2-",
    "HR-/HER2-",
    "HR+/HER2+",
    "HR-/HER2+",
    "Conflict",
    "Unknown"
]

print(
    collapsed["Molecular_subtype"]
    .value_counts(dropna=False)
    .reindex(molecular_order, fill_value=0)
)

print("\n================ RECEPTOR STATUS CONFLICT COUNTS ================")
for col in ["ER_Status", "PR_Status", "HER2_Status"]:
    n_conflict = (collapsed[col] == "Conflict").sum()
    print(f"{col}: {n_conflict}")

print("\n================ HISTOLOGY CONFLICT COUNT ================")
print((collapsed["Histologic.type"] == "Conflict").sum())


================ UNIQUE CASE SUMMARY ================
Number of unique Patient_BCR_Barcode: 171

================ HISTOLOGICAL SUBTYPE DISTRIBUTION ================
Ductal                      136
Lobular                      16
Mixed Ductal and Lobular      9
Missing                       6
Conflict                      4
Name: Histologic.type, dtype: Int64

================ MOLECULAR SUBTYPE DISTRIBUTION ================
HR+/HER2-     45
HR-/HER2-     13
HR+/HER2+      8
HR-/HER2+      1
Conflict     102
Unknown        2
Name: Molecular_subtype, dtype: int64

================ RECEPTOR STATUS CONFLICT COUNTS ================
ER_Status: 35
PR_Status: 85
HER2_Status: 45

================ HISTOLOGY CONFLICT COUNT ================
4


In [2]:
import pandas as pd
import re

# =====================================================
# Match unique case IDs between Metasheet_Analysis.xlsx
# and AUR_ClinicalData_20260427.NO.DATES.bloodID.withBCRv2.xlsx
# =====================================================

metasheet_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"

clinical_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUR_ClinicalData_20260427.NO.DATES.bloodID.withBCRv2.xlsx"


# -----------------------------
# Helper function: extract case ID
# Example:
# AUP-ANU1-NB1 -> AUP-ANU1
# AUP-AQ0Z     -> AUP-AQ0Z
# -----------------------------
def extract_case_id(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    m = re.search(r"(AUP-[A-Za-z0-9]+)", x)

    return m.group(1) if m else pd.NA


# -----------------------------
# Read Metasheet_Analysis.xlsx
# -----------------------------
meta = pd.read_excel(metasheet_file)

meta["Case_ID"] = meta["Collaborator_sample_id"].apply(extract_case_id)

meta_cases = set(meta["Case_ID"].dropna().unique())


# -----------------------------
# Read AUR clinical data file
# -----------------------------
clinical = pd.read_excel(clinical_file)

# In case duplicate Patient_BCR_Barcode columns exist,
# pandas may rename them as Patient_BCR_Barcode.1, etc.
barcode_cols = [
    c for c in clinical.columns
    if str(c).startswith("Patient_BCR_Barcode")
]

clinical_case_id_df = clinical[barcode_cols].apply(
    lambda col: col.apply(extract_case_id)
)

clinical["Case_ID"] = clinical_case_id_df.bfill(axis=1).iloc[:, 0]

clinical_cases = set(clinical["Case_ID"].dropna().unique())


# -----------------------------
# Match unique cases
# -----------------------------
matched_cases = meta_cases.intersection(clinical_cases)

meta_not_in_clinical = meta_cases - clinical_cases
clinical_not_in_meta = clinical_cases - meta_cases


# -----------------------------
# Print summary
# -----------------------------
print("================ CASE MATCH SUMMARY ================")
print(f"Unique cases in Metasheet_Analysis.xlsx: {len(meta_cases)}")
print(f"Unique cases in AUR clinical data file: {len(clinical_cases)}")
print(f"Number of matched unique cases: {len(matched_cases)}")
print(f"Cases in Metasheet but not in AUR clinical data file: {len(meta_not_in_clinical)}")
print(f"Cases in AUR clinical data file but not in Metasheet: {len(clinical_not_in_meta)}")

================ CASE MATCH SUMMARY ================
Unique cases in Metasheet_Analysis.xlsx: 270
Unique cases in AUR clinical data file: 171
Number of matched unique cases: 164
Cases in Metasheet but not in AUR clinical data file: 106
Cases in AUR clinical data file but not in Metasheet: 7


In [3]:
# =====================================================
# Print histology + molecular subtype summary
# ONLY for matched cases
# matched based on:
#   Metasheet_Analysis.xlsx: Collaborator_sample_id
#   AUR clinical file: Patient_BCR_Barcode
# =====================================================

cols_to_check = [
    "Histologic.type",
    "ER_Status",
    "PR_Status",
    "HER2_Status"
]

# -----------------------------------------------------
# Create clean Patient_BCR_Barcode column
# This handles possible duplicate barcode columns such as
# Patient_BCR_Barcode, Patient_BCR_Barcode.1, etc.
# -----------------------------------------------------
barcode_cols = [
    c for c in clinical.columns
    if str(c).startswith("Patient_BCR_Barcode")
]

clinical_barcode_clean = clinical[barcode_cols].bfill(axis=1).iloc[:, 0]

clinical_summary = clinical.copy()
clinical_summary["Patient_BCR_Barcode_clean"] = clinical_barcode_clean.astype("string").str.strip()

# Extract Case_ID from Patient_BCR_Barcode
clinical_summary["Case_ID"] = clinical_summary["Patient_BCR_Barcode_clean"].apply(extract_case_id)

# -----------------------------------------------------
# Keep only matched cases
# -----------------------------------------------------
clinical_matched = clinical_summary[
    clinical_summary["Case_ID"].isin(matched_cases)
].copy()

clinical_matched_sub = clinical_matched[
    ["Case_ID", "Patient_BCR_Barcode_clean"] + cols_to_check
].copy()

# Clean values
for col in ["Case_ID", "Patient_BCR_Barcode_clean"] + cols_to_check:
    clinical_matched_sub[col] = (
        clinical_matched_sub[col]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA})
    )

clinical_matched_sub = clinical_matched_sub.dropna(
    subset=["Patient_BCR_Barcode_clean"]
)

# -----------------------------------------------------
# Collapse values across repeated rows per Patient_BCR_Barcode
# -----------------------------------------------------
def collapse_patient_values(x):
    vals = sorted(x.dropna().unique())

    if len(vals) == 0:
        return "Missing"
    elif len(vals) == 1:
        return vals[0]
    else:
        return "Conflict"


def collect_patient_values(x):
    vals = sorted(x.dropna().unique())

    if len(vals) == 0:
        return "Missing"
    else:
        return "; ".join(vals)


collapsed = (
    clinical_matched_sub
    .groupby("Patient_BCR_Barcode_clean", as_index=False)
    .agg(
        Case_ID=("Case_ID", "first"),
        **{
            col: (col, collapse_patient_values)
            for col in cols_to_check
        },
        **{
            col + "_values": (col, collect_patient_values)
            for col in cols_to_check
        }
    )
)

# Rename for easier reading
collapsed = collapsed.rename(
    columns={"Patient_BCR_Barcode_clean": "Patient_BCR_Barcode"}
)

# -----------------------------------------------------
# Normalize receptor status
# -----------------------------------------------------
def normalize_receptor_status(x):
    if pd.isna(x) or x == "Missing":
        return "Missing"

    if x == "Conflict":
        return "Conflict"

    x_clean = str(x).strip().lower()

    if x_clean in ["positive", "pos", "+", "1", "yes"]:
        return "Positive"
    elif x_clean in ["negative", "neg", "-", "0", "no"]:
        return "Negative"
    else:
        return "Other"


for col in ["ER_Status", "PR_Status", "HER2_Status"]:
    collapsed[col + "_clean"] = collapsed[col].apply(normalize_receptor_status)

# -----------------------------------------------------
# Derive molecular subtype
# HR+ = ER+ OR PR+
# HR- = ER- AND PR-
# -----------------------------------------------------
def derive_molecular_subtype(row):
    er = row["ER_Status_clean"]
    pr = row["PR_Status_clean"]
    her2 = row["HER2_Status_clean"]

    if "Conflict" in [er, pr, her2]:
        return "Conflict"

    if er in ["Missing", "Other"] or pr in ["Missing", "Other"] or her2 in ["Missing", "Other"]:
        return "Unknown"

    if er == "Positive" or pr == "Positive":
        hr = "HR+"
    elif er == "Negative" and pr == "Negative":
        hr = "HR-"
    else:
        return "Unknown"

    if her2 == "Positive":
        her2_status = "HER2+"
    elif her2 == "Negative":
        her2_status = "HER2-"
    else:
        return "Unknown"

    return f"{hr}/{her2_status}"


collapsed["Molecular_subtype"] = collapsed.apply(
    derive_molecular_subtype,
    axis=1
)

# =====================================================
# Print required summaries
# =====================================================

print("\n================ MATCHED CASE SUMMARY ================")
print(f"Number of matched unique Patient_BCR_Barcode: {collapsed['Patient_BCR_Barcode'].nunique()}")

print("\n================ HISTOLOGICAL SUBTYPE DISTRIBUTION ================")
print(
    collapsed["Histologic.type"]
    .value_counts(dropna=False)
)

print("\n================ MOLECULAR SUBTYPE DISTRIBUTION ================")

molecular_order = [
    "HR+/HER2-",
    "HR-/HER2-",
    "HR+/HER2+",
    "HR-/HER2+",
    "Conflict",
    "Unknown"
]

print(
    collapsed["Molecular_subtype"]
    .value_counts(dropna=False)
    .reindex(molecular_order, fill_value=0)
)

print("\n================ RECEPTOR STATUS CONFLICT COUNTS ================")
for col in ["ER_Status", "PR_Status", "HER2_Status"]:
    print(f"{col}: {(collapsed[col] == 'Conflict').sum()}")

print("\n================ HISTOLOGY CONFLICT COUNT ================")
histology_conflict_count = (collapsed["Histologic.type"] == "Conflict").sum()
print(histology_conflict_count)

print("\n================ CASES WITH HISTOLOGY CONFLICT ================")

histology_conflict_cases = collapsed[
    collapsed["Histologic.type"] == "Conflict"
][
    [
        "Patient_BCR_Barcode",
        "Case_ID",
        "Histologic.type_values",
        "ER_Status",
        "PR_Status",
        "HER2_Status",
        "Molecular_subtype"
    ]
]

if histology_conflict_cases.empty:
    print("No histology conflict cases found.")
else:
    print(histology_conflict_cases.to_string(index=False))


================ MATCHED CASE SUMMARY ================
Number of matched unique Patient_BCR_Barcode: 164

================ HISTOLOGICAL SUBTYPE DISTRIBUTION ================
Ductal                      130
Lobular                      16
Mixed Ductal and Lobular      9
Missing                       5
Conflict                      4
Name: Histologic.type, dtype: Int64

================ MOLECULAR SUBTYPE DISTRIBUTION ================
HR+/HER2-     44
HR-/HER2-     11
HR+/HER2+      7
HR-/HER2+      1
Conflict     100
Unknown        1
Name: Molecular_subtype, dtype: int64

================ RECEPTOR STATUS CONFLICT COUNTS ================
ER_Status: 34
PR_Status: 83
HER2_Status: 44

================ HISTOLOGY CONFLICT COUNT ================
4

================ CASES WITH HISTOLOGY CONFLICT ================
Patient_BCR_Barcode  Case_ID           Histologic.type_values ER_Status PR_Status HER2_Status Molecular_subtype
           AUP-ANF8 AUP-ANF8                  Ductal; Lobular  Positive  

In [5]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# =====================================================
# Input files
# =====================================================
metasheet_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"

clinical_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUR_ClinicalData_20260427.NO.DATES.bloodID.withBCRv2.xlsx"

out_dir = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data")

out_filled_metasheet = out_dir / "Metasheet_Analysis.filled_with_AUR_clinicalData_20260427.xlsx"
out_patient_annotation = out_dir / "AUR_patient_level_collapsed_annotation.xlsx"
out_histology_conflict = out_dir / "AUR_histology_conflict_cases.xlsx"
out_receptor_conflict = out_dir / "AUR_receptor_conflict_cases.xlsx"


# =====================================================
# Helper functions
# =====================================================

def extract_case_id(x):
    """
    Extract AURORA case ID.
    Example:
    AUP-ANU1-NB1 -> AUP-ANU1
    AUP-AQ0Z     -> AUP-AQ0Z
    """
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()
    m = re.search(r"(AUP-[A-Za-z0-9]+)", x)

    return m.group(1) if m else pd.NA


def clean_value(x):
    """
    Convert blank / NA-like values to pandas NA.
    """
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x.lower() in ["", "na", "n/a", "nan", "none", "null"]:
        return pd.NA

    return x


def is_empty_series(s):
    """
    Identify empty cells in metasheet.
    Used so we only fill empty cells and do not overwrite existing values.
    """
    return (
        s.isna()
        | s.astype("string").str.strip().str.lower().isin(
            ["", "na", "n/a", "nan", "none", "null"]
        )
    )


def collapse_patient_values(x):
    """
    Collapse repeated rows per Patient_BCR_Barcode.

    Rule:
    - all missing          -> Missing
    - one unique value     -> keep value
    - multiple values      -> Conflict
    """
    vals = sorted(pd.Series(x).dropna().astype(str).str.strip().unique())

    if len(vals) == 0:
        return "Missing"
    elif len(vals) == 1:
        return vals[0]
    else:
        return "Conflict"


def collect_patient_values(x):
    """
    Keep all unique non-missing values for audit/conflict checking.
    """
    vals = sorted(pd.Series(x).dropna().astype(str).str.strip().unique())

    if len(vals) == 0:
        return "Missing"
    else:
        return "; ".join(vals)


def normalize_receptor_status(x):
    """
    Normalize receptor status to Positive / Negative / Missing / Conflict / Other.
    """
    if pd.isna(x) or x == "Missing":
        return "Missing"

    if x == "Conflict":
        return "Conflict"

    x_clean = str(x).strip().lower()

    if x_clean in ["positive", "pos", "+", "1", "yes"]:
        return "Positive"
    elif x_clean in ["negative", "neg", "-", "0", "no"]:
        return "Negative"
    else:
        return "Other"


def derive_molecular_subtype_from_values(er, pr, her2):
    """
    Molecular subtype rule:
    HR+ = ER+ or PR+
    HR- = ER- and PR-
    """
    er = normalize_receptor_status(er)
    pr = normalize_receptor_status(pr)
    her2 = normalize_receptor_status(her2)

    if "Conflict" in [er, pr, her2]:
        return "Conflict"

    if er in ["Missing", "Other"] or pr in ["Missing", "Other"] or her2 in ["Missing", "Other"]:
        return "Missing"

    if er == "Positive" or pr == "Positive":
        hr = "HR+"
    elif er == "Negative" and pr == "Negative":
        hr = "HR-"
    else:
        return "Missing"

    if her2 == "Positive":
        her2_status = "HER2+"
    elif her2 == "Negative":
        her2_status = "HER2-"
    else:
        return "Missing"

    return f"{hr}/{her2_status}"


# =====================================================
# Load files
# =====================================================
meta = pd.read_excel(metasheet_file)
clinical = pd.read_excel(clinical_file)


# =====================================================
# Extract Case_ID from metasheet
# =====================================================
meta["Case_ID"] = meta["Collaborator_sample_id"].apply(extract_case_id)

meta_cases = set(meta["Case_ID"].dropna().unique())


# =====================================================
# Use only required AUR clinical columns
# =====================================================
clinical_cols = [
    "Histologic.type",
    "ER_Status",
    "PR_Status",
    "HER2_Status"
]

# Handle possible duplicate Patient_BCR_Barcode columns
barcode_cols = [
    c for c in clinical.columns
    if str(c).startswith("Patient_BCR_Barcode")
]

if len(barcode_cols) == 0:
    raise ValueError("No Patient_BCR_Barcode column found in clinical file.")

clinical_sub = clinical[barcode_cols + clinical_cols].copy()

# Make one clean Patient_BCR_Barcode column
clinical_sub["Patient_BCR_Barcode"] = (
    clinical_sub[barcode_cols]
    .bfill(axis=1)
    .iloc[:, 0]
)

clinical_sub = clinical_sub[["Patient_BCR_Barcode"] + clinical_cols].copy()


# =====================================================
# Clean Patient_BCR_Barcode and clinical values
# =====================================================
for col in ["Patient_BCR_Barcode"] + clinical_cols:
    clinical_sub[col] = clinical_sub[col].apply(clean_value)

clinical_sub = clinical_sub.dropna(subset=["Patient_BCR_Barcode"])

clinical_sub["Case_ID"] = clinical_sub["Patient_BCR_Barcode"].apply(extract_case_id)

clinical_cases = set(clinical_sub["Case_ID"].dropna().unique())


# =====================================================
# Match cases between metasheet and AUR clinical file
# =====================================================
matched_cases = meta_cases.intersection(clinical_cases)

meta_not_in_clinical = meta_cases - clinical_cases
clinical_not_in_meta = clinical_cases - meta_cases


print("================ CASE MATCH SUMMARY ================")
print(f"Unique cases in Metasheet_Analysis.xlsx: {len(meta_cases)}")
print(f"Unique cases in AUR clinical data file: {len(clinical_cases)}")
print(f"Number of matched unique cases: {len(matched_cases)}")
print(f"Cases in Metasheet but not in AUR clinical data file: {len(meta_not_in_clinical)}")
print(f"Cases in AUR clinical data file but not in Metasheet: {len(clinical_not_in_meta)}")


# =====================================================
# Keep only matched clinical cases
# =====================================================
clinical_matched = clinical_sub[
    clinical_sub["Case_ID"].isin(matched_cases)
].copy()


# =====================================================
# Collapse repeated rows per Patient_BCR_Barcode
# This is the most important patient-level step
# =====================================================
patient_level = (
    clinical_matched
    .groupby("Patient_BCR_Barcode", as_index=False)
    .agg(
        Case_ID=("Case_ID", "first"),

        Histological_subtype=("Histologic.type", collapse_patient_values),
        ER_Status=("ER_Status", collapse_patient_values),
        PR_Status=("PR_Status", collapse_patient_values),
        HER2_Status=("HER2_Status", collapse_patient_values),

        Histological_subtype_values=("Histologic.type", collect_patient_values),
        ER_Status_values=("ER_Status", collect_patient_values),
        PR_Status_values=("PR_Status", collect_patient_values),
        HER2_Status_values=("HER2_Status", collect_patient_values),
    )
)


# =====================================================
# Derive molecular subtype at patient level
# =====================================================
patient_level["Molecular_subtype"] = patient_level.apply(
    lambda row: derive_molecular_subtype_from_values(
        row["ER_Status"],
        row["PR_Status"],
        row["HER2_Status"]
    ),
    axis=1
)


# =====================================================
# Print case-level distributions from collapsed patient-level data
# =====================================================
print("\n================ CASE-LEVEL HISTOLOGICAL SUBTYPE DISTRIBUTION ================")
print(patient_level["Histological_subtype"].value_counts(dropna=False))

print("\n================ CASE-LEVEL MOLECULAR SUBTYPE DISTRIBUTION ================")
molecular_order = [
    "HR+/HER2-",
    "HR-/HER2-",
    "HR+/HER2+",
    "HR-/HER2+",
    "Conflict",
    "Missing"
]

print(
    patient_level["Molecular_subtype"]
    .value_counts(dropna=False)
    .reindex(molecular_order, fill_value=0)
)

print("\n================ RECEPTOR STATUS CONFLICT COUNTS ================")
for col in ["ER_Status", "PR_Status", "HER2_Status"]:
    print(f"{col}: {(patient_level[col] == 'Conflict').sum()}")

print("\n================ HISTOLOGY CONFLICT COUNT ================")
histology_conflict_count = (patient_level["Histological_subtype"] == "Conflict").sum()
print(histology_conflict_count)


# =====================================================
# Print cases with histology conflict
# =====================================================
histology_conflict_cases = patient_level[
    patient_level["Histological_subtype"] == "Conflict"
][
    [
        "Patient_BCR_Barcode",
        "Case_ID",
        "Histological_subtype_values",
        "ER_Status",
        "PR_Status",
        "HER2_Status",
        "Molecular_subtype"
    ]
]

print("\n================ CASES WITH HISTOLOGY CONFLICT ================")

if histology_conflict_cases.empty:
    print("No histology conflict cases found.")
else:
    print(histology_conflict_cases.to_string(index=False))


# =====================================================
# Print cases with receptor conflicts
# =====================================================
receptor_conflict_cases = patient_level[
    (patient_level["ER_Status"] == "Conflict")
    | (patient_level["PR_Status"] == "Conflict")
    | (patient_level["HER2_Status"] == "Conflict")
][
    [
        "Patient_BCR_Barcode",
        "Case_ID",
        "ER_Status_values",
        "PR_Status_values",
        "HER2_Status_values",
        "Molecular_subtype"
    ]
]

print("\n================ CASES WITH RECEPTOR STATUS CONFLICT ================")

if receptor_conflict_cases.empty:
    print("No receptor conflict cases found.")
else:
    print(receptor_conflict_cases.to_string(index=False))


# =====================================================
# Merge patient-level annotation into metasheet
# Fill only empty cells
# Do not overwrite existing values
# =====================================================
annotation_for_merge = patient_level[
    [
        "Case_ID",
        "Histological_subtype",
        "ER_Status",
        "PR_Status",
        "HER2_Status",
        "Molecular_subtype"
    ]
].copy()

annotation_for_merge = annotation_for_merge.rename(
    columns={
        "Histological_subtype": "AUR_Histological_subtype",
        "ER_Status": "AUR_ER_Status",
        "PR_Status": "AUR_PR_Status",
        "HER2_Status": "AUR_HER2_Status",
        "Molecular_subtype": "AUR_Molecular_subtype"
    }
)

meta_filled = meta.merge(
    annotation_for_merge,
    on="Case_ID",
    how="left"
)


# =====================================================
# Ensure target columns exist in metasheet
# =====================================================
target_cols = [
    "Histological_subtype",
    "ER_Status",
    "PR_Status",
    "HER2_Status",
    "Molecular_subtype"
]

for col in target_cols:
    if col not in meta_filled.columns:
        meta_filled[col] = pd.NA


# =====================================================
# Fill only empty cells from AUR annotation
# =====================================================
fill_map = {
    "Histological_subtype": "AUR_Histological_subtype",
    "ER_Status": "AUR_ER_Status",
    "PR_Status": "AUR_PR_Status",
    "HER2_Status": "AUR_HER2_Status",
    "Molecular_subtype": "AUR_Molecular_subtype"
}

for target_col, source_col in fill_map.items():
    mask_fill = is_empty_series(meta_filled[target_col]) & meta_filled[source_col].notna()
    meta_filled.loc[mask_fill, target_col] = meta_filled.loc[mask_fill, source_col]


# =====================================================
# Derive Molecular_subtype again from final filled receptor columns
# but only where Molecular_subtype is empty
# =====================================================
derived_final_molecular = meta_filled.apply(
    lambda row: derive_molecular_subtype_from_values(
        row["ER_Status"],
        row["PR_Status"],
        row["HER2_Status"]
    ),
    axis=1
)

mask_molecular_empty = is_empty_series(meta_filled["Molecular_subtype"])
meta_filled.loc[mask_molecular_empty, "Molecular_subtype"] = derived_final_molecular[mask_molecular_empty]


# =====================================================
# Drop temporary AUR helper columns before saving final metasheet
# =====================================================
helper_cols = [
    "AUR_Histological_subtype",
    "AUR_ER_Status",
    "AUR_PR_Status",
    "AUR_HER2_Status",
    "AUR_Molecular_subtype"
]

meta_filled_final = meta_filled.drop(columns=helper_cols)


# =====================================================
# Print sample-level distribution from filled metasheet
# Sample-level = each row in Metasheet_Analysis
# =====================================================
matched_meta_filled = meta_filled_final[
    meta_filled_final["Case_ID"].isin(matched_cases)
].copy()

print("\n================ SAMPLE-LEVEL HISTOLOGICAL SUBTYPE DISTRIBUTION ================")
print(
    matched_meta_filled["Histological_subtype"]
    .value_counts(dropna=False)
)

print("\n================ SAMPLE-LEVEL MOLECULAR SUBTYPE DISTRIBUTION ================")
print(
    matched_meta_filled["Molecular_subtype"]
    .value_counts(dropna=False)
    .reindex(molecular_order, fill_value=0)
)


# =====================================================
# Save outputs
# =====================================================
patient_level.to_excel(out_patient_annotation, index=False)
histology_conflict_cases.to_excel(out_histology_conflict, index=False)
receptor_conflict_cases.to_excel(out_receptor_conflict, index=False)
meta_filled_final.to_excel(out_filled_metasheet, index=False)

print("\n================ OUTPUT FILES SAVED ================")
print(f"Patient-level collapsed annotation:")
print(out_patient_annotation)

print(f"\nHistology conflict cases:")
print(out_histology_conflict)

print(f"\nReceptor conflict cases:")
print(out_receptor_conflict)

print(f"\nFilled metasheet:")
print(out_filled_metasheet)

================ CASE MATCH SUMMARY ================
Unique cases in Metasheet_Analysis.xlsx: 270
Unique cases in AUR clinical data file: 171
Number of matched unique cases: 164
Cases in Metasheet but not in AUR clinical data file: 106
Cases in AUR clinical data file but not in Metasheet: 7

================ CASE-LEVEL HISTOLOGICAL SUBTYPE DISTRIBUTION ================
Ductal                      130
Lobular                      16
Mixed Ductal and Lobular      9
Missing                       5
Conflict                      4
Name: Histological_subtype, dtype: int64

================ CASE-LEVEL MOLECULAR SUBTYPE DISTRIBUTION ================
HR+/HER2-     44
HR-/HER2-     11
HR+/HER2+      7
HR-/HER2+      1
Conflict     100
Missing        1
Name: Molecular_subtype, dtype: int64

================ RECEPTOR STATUS CONFLICT COUNTS ================
ER_Status: 34
PR_Status: 83
HER2_Status: 44

================ HISTOLOGY CONFLICT COUNT ================
4

================ CASES WITH HISTOLOG